# 🧠 3D VisReg JEPA Self-Supervised Pre-training on Colab GPU
### BraTS 2024 Adult Glioma Volumetric Pre-training Suite (Google Colab)

This notebook executes self-supervised 3D VisReg JEPA pre-training on **Google Colab with a GPU (L4 / A100)**.

**Key Features & Parity with Kaggle Suite**:
1. **Full Algorithmic Parity**: Matches `notebooks/01_train_visreg_3d.ipynb` with explicit, modular CLI argument control.
2. **High-Resolution $8^3$ Patch Size**: Enables fine-grained voxel tokenization ($16^3 = 4,096$ tokens vs standard $8^3 = 512$ tokens) leveraging Colab GPU memory and FlashAttention.
3. **Anisotropic 3D Target Jitter Masking**: Auto-scales across 6 multi-aspect ratio target cuboids (776 target tokens, 1,536 context tokens at 37.5%).
4. **Dual Cosine Schedulers**: Cosine learning rate decay with linear warmup ($5 \times 10^{-4} \to 1 \times 10^{-5}$) paired with cosine regularizer annealing ($1.0 \to 0.10$ after 15 warmup epochs).
5. **GPU Throughput**: Optimized `batch_size=16`, AMP FP16, and multi-worker CPU prefetching.
6. **Google Drive Checkpoint Streaming**: Persists checkpoints and logs directly to `/content/drive/MyDrive/checkpoints/` so no work is lost on disconnection.


## 1. Hardware & CUDA Environment Verification


In [ ]:
import datetime
import time
import torch

NOTEBOOK_START_TIME = time.time()
NOTEBOOK_START_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"⏱️ Session Start Time: {NOTEBOOK_START_STR}")

!nvidia-smi

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device Name:      {gpu_name}")
    print(f"Total VRAM:       {vram_gb:.2f} GB")
    print(f"✓ GPU active: {gpu_name} ({vram_gb:.2f} GB VRAM). Ready for 3D volumetric training.")
else:
    raise RuntimeError("No GPU detected! Go to Runtime -> Change runtime type -> Hardware accelerator -> GPU.")


## 2. Mount Google Drive (Checkpoint Safety)
Mounting Google Drive ensures that model checkpoints, training logs, and exported weights are saved permanently and survive session disconnects.


In [ ]:
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    try:
        drive.mount("/content/drive")
    except Exception:
        drive.mount("/content/drive", force_remount=True)
else:
    print("✓ Google Drive is already mounted.")

# Standard thesis output directory on Google Drive
thesis_dir = Path("/content/drive/MyDrive/thesis")
output_dir = thesis_dir / "pretrain model"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✓ Checkpoints and logs will stream directly to Google Drive:\n  {output_dir}")


## 3. Experiment Configuration & Reproducibility Parameters


In [ ]:
# =========================================================================
# ⚙️ Experiment Configuration & Reproducibility Parameters
# =========================================================================

SEED = 42                 # Deterministic seed for weight init, augmentations, and data splits
NUM_WORKERS = 4          # Parallel CPU data loading workers (Colab provides multi-vCPU CPU allocation)
PATCH_SIZE = 8           # High-resolution 8³ voxel patch size for high-res model (4,096 token grid; default 16 for standard)
PRETRAIN_BATCH_SIZE = 32  # High-throughput volumetric batch size
EPOCHS = 200             # Pre-training epochs (200-epoch schedule at B=32)
LEARNING_RATE = 5e-4     # Base learning rate with linear warmup + cosine decay
WEIGHT_DECAY = 0.05      # AdamW weight decay
REG_WARMUP_EPOCHS = 30   # Warmup epochs with lambda_reg=1.0 before cosine annealing (30 epochs)
REG_MIN_SCALE = 0.10     # Minimum regularizer scale (cosine annealed from 1.0 down to 0.10)


## 4. Dependencies Installation


In [ ]:
!pip install -q --no-cache-dir monai nibabel tabulate matplotlib
import monai
import nibabel as nib
import tabulate

print(f"✓ MONAI Version:    v{monai.__version__}")
print(f"✓ NiBabel Version:  v{nib.__version__}")
print(f"✓ Tabulate Version: v{tabulate.__version__}")


## 5. Codebase Setup & Editable Installation


In [ ]:
import os
import shutil
import sys
from pathlib import Path

REPO_URL = "https://github.com/hanriman/tumor_segmentation_3d.git"
work_dir = Path("/content/tumor_segmentation_3d")

# Clean up incomplete clone if needed
if work_dir.exists() and not (work_dir / "src" / "brats_jepa_3d").exists():
    print("⚠️ Cleaning up incomplete repository clone...")
    shutil.rmtree(work_dir)

if not (work_dir / "src" / "brats_jepa_3d").exists():
    print(f"Cloning codebase from: {REPO_URL} ...")
    !git clone {REPO_URL} {work_dir}
else:
    print(f"Updating codebase to latest main from: {REPO_URL} ...")
    !git -C {work_dir} pull origin main

os.chdir(str(work_dir))
%cd {work_dir}

src_dir = work_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

!pip install -q -e .

print(f"\n✓ Working directory set to: {Path.cwd()}")
!git log -1 --oneline 2>/dev/null || echo "(Git commit info unavailable)"

!git -C {work_dir} checkout main
!git -C {work_dir} pull --ff-only origin main
assert (Path(work_dir) / "src/brats_jepa_3d/metrics/regions.py").exists(), \
    "Branch checkout failed: regions.py missing — aborting."
print("On branch main:", end=" ")
!git -C {work_dir} log -1 --oneline


## Dataset Ingestion & Discovery
Extracts the preprocessed BraTS 3D dataset from Google Drive (`My Drive/thesis/data/`) into Colab fast local NVMe storage (`/content/data/`) and validates data integrity.

> **Directory Structure**: The `.zip` extracts to root folder `brats_gli_3d_v2/` containing `metadata.csv` and multi-modal `.npz` volumes.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
from brats_jepa_3d.config import get_dataset_dir, get_metadata_path
from brats_jepa_3d.data import BraTS3DDataset

# 1. Unpack dataset from Google Drive (fast local NVMe extraction)
thesis_dir = Path("/content/drive/MyDrive/thesis")
data_drive_dir = thesis_dir / "data"
target_data_dir = Path("/content/data/brats_gli_3d_v2")

if not (target_data_dir / "metadata.csv").exists() and not Path("/content/data/metadata.csv").exists():
    zip_candidates = []
    if data_drive_dir.exists():
        zip_candidates = sorted(data_drive_dir.glob("*.zip"))
    if not zip_candidates:
        zip_candidates = sorted(Path("/content/drive/MyDrive").glob("*brats*.zip"))

    if zip_candidates:
        dataset_zip = zip_candidates[0]
        print(f"✓ Found dataset archive on Google Drive: {dataset_zip}")
        print("  Unpacking to /content/data/ (fast local NVMe SSD) ...")
        !mkdir -p /content/data
        !unzip -q "{dataset_zip}" -d /content/data/
        print("✓ Dataset extracted successfully.")
    elif data_drive_dir.exists() and any((data_drive_dir / sub / "metadata.csv").exists() for sub in ["brats_gli_3d_v2", "."]):
        print(f"✓ Found pre-extracted dataset in {data_drive_dir}.")
        source_folder = data_drive_dir / "brats_gli_3d_v2" if (data_drive_dir / "brats_gli_3d_v2").exists() else data_drive_dir
        !mkdir -p /content/data
        !cp -rs "{source_folder}" /content/data/brats_gli_3d_v2
    else:
        raise FileNotFoundError(
            f"❌ No dataset .zip found in {data_drive_dir}!\n"
            f"Please upload your dataset .zip (containing 'brats_gli_3d_v2') into 'My Drive/thesis/data/'."
        )

# 2. Discover dataset directory (points to brats_gli_3d_v2)
candidates = [
    Path("/content/data/brats_gli_3d_v2"),
    Path("/content/data"),
    Path("/content/brats_gli_3d_v2"),
]
data_dir = next((c for c in candidates if (c / "metadata.csv").exists()), None)
if data_dir is None:
    matches = list(Path("/content/data").glob("**/metadata.csv"))
    if matches:
        data_dir = matches[0].parent
    else:
        raise FileNotFoundError("❌ metadata.csv not found after extraction in /content/data/!")

meta_path = data_dir / "metadata.csv"
os.environ["BRATS3D_DATA_DIR"] = str(data_dir)

print(f"\n=== DATASET DISCOVERY ===")
print(f"Dataset Directory: {data_dir} (Exists: {data_dir.exists()})")
print(f"Metadata CSV:      {meta_path} (Exists: {meta_path.exists()})")

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f"\n✓ Loaded Metadata: {len(df)} total records across splits:")
    print(df["split"].value_counts().to_string())
    assert "has_et" in df.columns, "metadata.csv lacks has_et — attach v2 integer mask dataset."

    ds = BraTS3DDataset(data_dir=data_dir, split="train")
    sample = ds[0]
    print("\n✓ Sample Tensor Verification:")
    print(f"  Image Shape: {sample['image'].shape} (dtype: {sample['image'].dtype})")
    print(f"  Mask Shape:  {sample['mask'].shape} (dtype: {sample['mask'].dtype})")
    print(f"  Tumor Voxels: {int((sample['mask'] > 0).sum()):,}")
else:
    raise FileNotFoundError(f"Metadata CSV not found at {meta_path}. Please check your dataset zip.")


## 7. Pre-Flight Verification Smoke Test (Optional, 2 Minutes)
Verifies CUDA, AMP, masking transform, and loss functions on 2 mini-batches with `--patch_size {PATCH_SIZE}` before starting the full pre-training run.


In [ ]:
# Fallback defaults if configuration cell was skipped
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
REG_MIN_SCALE = globals().get('REG_MIN_SCALE', 0.10)

!python scripts/train_jepa_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --smoke_test \
    --reg_warmup_epochs 2 \
    --reg_min_scale {REG_MIN_SCALE}


## 8. Full 3D VisReg JEPA Self-Supervised Pre-training (Colab GPU)
Executes full self-supervised 3D VisReg JEPA pre-training with explicit CLI arguments:
- **Patch size**: `--patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE}` ($8^3$ voxel patches, $4,096$ tokens)
- **Batch size 32** with automatic mixed precision (`--amp`)
- **Regularizer schedule**: 30-epoch linear warmup to constant $\lambda_{\text{reg}} = 1.0$ (`reg_warmup_epochs=30`, `reg_min_scale=1.0`)
- Checkpoints and logs stream directly to Google Drive


In [ ]:
# Fallback defaults if configuration cell was skipped
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
PRETRAIN_BATCH_SIZE = globals().get('PRETRAIN_BATCH_SIZE', 32)
EPOCHS = globals().get('EPOCHS', 200)
SEED = globals().get('SEED', 42)
NUM_WORKERS = globals().get('NUM_WORKERS', 4)
LEARNING_RATE = globals().get('LEARNING_RATE', 5e-4)
WEIGHT_DECAY = globals().get('WEIGHT_DECAY', 0.05)
REG_WARMUP_EPOCHS = globals().get('REG_WARMUP_EPOCHS', 30)
REG_MIN_SCALE = globals().get('REG_MIN_SCALE', 0.10)

!python scripts/train_jepa_3d.py \
    --data_dir "{data_dir}" \
    --output_dir "{output_dir}" \
    --model_type visreg_jepa \
    --patch_size {PATCH_SIZE} {PATCH_SIZE} {PATCH_SIZE} \
    --seed {SEED} \
    --epochs {EPOCHS} \
    --batch_size {PRETRAIN_BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --learning_rate {LEARNING_RATE} \
    --weight_decay {WEIGHT_DECAY} \
    --reg_warmup_epochs {REG_WARMUP_EPOCHS} \
    --reg_min_scale {REG_MIN_SCALE} \
    --amp


## 9. Pre-training Diagnostics: Brain-Aware Masking Check


In [ ]:
PATCH_SIZE = globals().get('PATCH_SIZE', 8)
!python scripts/diagnose_visreg_3d.py \
    --data_dir "{data_dir}" \
    --patch_size {PATCH_SIZE} \
    --num_volumes 200


## 10. Export & Package Pre-training Artifacts to Google Drive
Packages pre-trained weights, logs, and diagnostic reports into a verified zip archive in Google Drive.


In [ ]:
import datetime
import time
from pathlib import Path
from brats_jepa_3d.utils import export_artifacts

# 1. Package checkpoint and logs directly to Google Drive
zip_dest = output_dir / "visreg_pretrain_outputs.zip"
export_res = export_artifacts(
    export_name="visreg_pretrain_outputs",
    export_dir=Path("/content/export_visreg_pretrain"),
    zip_path=zip_dest,
    source_dir=output_dir,
    model_prefix="visreg",
    verbose=True,
)

# 2. Session Timing Report
NOTEBOOK_END_TIME = time.time()
NOTEBOOK_END_STR = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
start_time = globals().get("NOTEBOOK_START_TIME", NOTEBOOK_END_TIME)
total_elapsed_sec = NOTEBOOK_END_TIME - start_time
hours, rem = divmod(total_elapsed_sec, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 50)
print(f"⏱️ Session Start Time:   {globals().get('NOTEBOOK_START_STR', 'N/A')}")
print(f"⏱️ Session End Time:     {NOTEBOOK_END_STR}")
print(f"⏱️ Total Execution Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s ({total_elapsed_sec:.2f}s)")
print("=" * 50)
print(f"✓ Checkpoint and diagnostic package saved to Google Drive:\n  {zip_dest}")
